In [31]:
import numpy as np
import pandas as pd
import os
import re
from numpy.linalg import norm

In [32]:
OHCO = ['book_title', 'chap_num', 'para_num', 'sent_num', 'token_num']

In [33]:
Corpus_Tokens=pd.read_csv("Corpus_Tokens.csv").set_index(OHCO)

In [34]:
Corpus_Tokens

token_str  \
book_title             chap_num para_num sent_num token_num             
The Old Curiosity Shop 1        0        0        0          Although   
                                                  1                 I   
                                                  2                am   
                                                  3                an   
                                                  4               old   
...                                                               ...   
Barnaby Rudge          82       20       1        20          talking   
                                                  21               to   
                                                  22              the   
                                                  23          present   
                                                  24             time   

                                                             term_str  \
book_title             chap_num para_num sent_num token_num             
The Old Curiosity Shop 1        0        0        0          although   
                                                  1                 i   
                                                  2                am   
                                                  3                an   
                                                  4               old   
...                                                               ...   
Barnaby Rudge          82       20       1        20          talking   
                                                  21               to   
                                                  22              the   
                                                  23          present   
                                                  24             time   

                                                                      pos_tuple  \
book_title             chap_num para_num sent_num token_num                       
The Old Curiosity Shop 1        0        0        0          ('Although', 'IN')   
                                                  1                ('I', 'PRP')   
                                                  2               ('am', 'VBP')   
                                                  3                ('an', 'DT')   
                                                  4               ('old', 'JJ')   
...                                                                         ...   
Barnaby Rudge          82       20       1        20         ('talking', 'VBG')   
                                                  21               ('to', 'TO')   
                                                  22              ('the', 'DT')   
                                                  23          ('present', 'JJ')   
                                                  24             ('time', 'NN')   

                                                             pos pos_group  
book_title             chap_num para_num sent_num token_num                 
The Old Curiosity Shop 1        0        0        0           IN        IN  
                                                  1          PRP        PR  
                                                  2          VBP        VB  
                                                  3           DT        DT  
                                                  4           JJ        JJ  
...                                                          ...       ...  
Barnaby Rudge          82       20       1        20         VBG        VB  
                                                  21          TO        TO  
                                                  22          DT        DT  
                                                  23          JJ        JJ  
                                                  24          NN        NN  

[3857375 rows x 5 columns]

In [35]:
def bow(corpus,bag_choice="CHAPS"):
    OHCO = ['book_title', 'chap_num', 'para_num', 'sent_num', 'token_num']
    bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1])
    bag = bag_choice
    BOW = corpus.groupby(bags[bag]+['term_str']).term_str.count().to_frame('n') 
    return BOW

In [36]:
def tfidf(BOW,tf_method="sum",idf_method="standard"):
    DTCM = BOW.n.unstack(fill_value=0)
    
    if tf_method == 'sum':
        TF = DTCM.T / DTCM.T.sum()

    elif tf_method == 'smooth':
        TF = (DTCM.T / DTCM.T.sum()) + 1

    elif tf_method == 'max':
        TF = DTCM.T / DTCM.T.max()
    
    elif tf_method == 'log':
        TF = np.log2(1 + DTCM.T)
    
    elif tf_method == 'raw':
        TF = DTCM.T
    
    elif tf_method == 'double_norm':
        TF = DTCM.T / DTCM.T.max()
    
    elif tf_method == 'binary':
        TF = DTCM.T.astype('bool').astype('int')
    
    TF = TF.T

    DF = DTCM.astype('bool').sum() 

    N = DTCM.shape[0]

    if idf_method == 'standard':
        IDF = np.log2(N / DF)

    elif idf_method == 'max':
        IDF = np.log2(DF.max() / DF) 

    elif idf_method == 'plus':
        IDF = np.log2(N / DF) + 1

    elif idf_method == 'smooth':
        IDF = np.log2((1 + N) / (1 + DF)) + 1

    TFIDF = TF * IDF
    return TFIDF, DTCM
    

In [37]:
BOW=bow(Corpus_Tokens,bag_choice="BOOKS")

In [38]:
TFIDF, DTM=tfidf(BOW)

In [39]:
BOW

n
book_title             term_str      
A Tale of Two Cities 1 a          422
                       abandoned    2
                       abiding      1
                       ability      1
                       about       23
...                               ...
The Old Curiosity Shop youve       18
                       zambullo     1
                       zeal         2
                       zenith       1
                       zest         1

[233588 rows x 1 columns]

In [40]:
DTM

term_str,0,000,1,12,12d,13,14th,1500,1747,1757,...,zest,zests,zig,zigzag,zigzagged,zone,zoo,zoodle,zooks,zounds
book_title,,,,,,,,,,,,,,,,,,,,,
A Tale of Two Cities 1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A Tale of Two Cities 2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
A Tale of Two Cities 3,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
Barnaby Rudge,0,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,0,0,0,1
Bleak House,0,0,1,0,0,0,0,0,0,0,...,1,0,1,0,0,2,0,1,0,0
David Copperfield,0,0,0,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
Dombey and Son,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
Great Expectations,0,0,1,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
Hard Times 1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [41]:
TFIDF

term_str,0,000,1,12,12d,13,14th,1500,1747,1757,...,zest,zests,zig,zigzag,zigzagged,zone,zoo,zoodle,zooks,zounds
book_title,,,,,,,,,,,,,,,,,,,,,
A Tale of Two Cities 1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A Tale of Two Cities 2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A Tale of Two Cities 3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000093,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Barnaby Rudge,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000005,0.00000,0.000010,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000018
Bleak House,0.000000,0.000000,0.000006,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000004,0.00000,0.000007,0.000000,0.000000,0.000020,0.000000,0.000013,0.000000,0.000000
David Copperfield,0.000000,0.000000,0.000000,0.000000,0.000013,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.00000,0.000000,0.000008,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Dombey and Son,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.00001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Great Expectations,0.000000,0.000000,0.000012,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000007,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Hard Times 1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [42]:
tfidf_L2 = TFIDF.apply(lambda x: x / norm(x), 1)
tfidf_L2_filter = tfidf_L2.copy()

tfidf_L2_filter[tfidf_L2_filter < 0.05] = 0

tfidf_L2_reduced = tfidf_L2_filter.loc[:, (tfidf_L2_filter!= 0).any()]
tfidf_L2_reduced

term_str,abel,ada,affery,agnes,allan,allen,amy,antoine,arthur,ath,...,willet,windlass,winkle,wold,woodcourt,woodman,wopsle,wrayburn,wren,yo
book_title,,,,,,,,,,,,,,,,,,,,,
A Tale of Two Cities 1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.055247,0.000000,0.00000,0.000000,0.000000
A Tale of Two Cities 2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.077648,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
A Tale of Two Cities 3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.058555,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
Barnaby Rudge,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.218013,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
Bleak House,0.000000,0.307637,0.000000,0.000000,0.069385,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.065768,0.119544,0.000000,0.000000,0.00000,0.000000,0.000000
David Copperfield,0.000000,0.000000,0.000000,0.219673,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
Dombey and Son,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
Great Expectations,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.145311,0.00000,0.000000,0.000000
Hard Times 1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000


In [43]:
BOW = BOW.join(TFIDF.stack().to_frame('tfidf'), how='left')

In [44]:
BOW

n     tfidf
book_title             term_str                
A Tale of Two Cities 1 a          422  0.000000
                       abandoned    2  0.000015
                       abiding      1  0.000077
                       ability      1  0.000025
                       about       23  0.000000
...                               ...       ...
The Old Curiosity Shop youve       18  0.000011
                       zambullo     1  0.000021
                       zeal         2  0.000009
                       zenith       1  0.000008
                       zest         1  0.000006

[233588 rows x 2 columns]

In [45]:
BOW.to_csv("BOW.csv", index=True)
DTM.to_csv("DTM.csv", index=True)
TFIDF.to_csv("TFIDF.csv", index=True)
tfidf_L2_reduced.to_csv("tfidf_L2.csv", index=True)